# Build model — number + percent (all_open groups) features, OLD

Trains the same lightweight pipeline as `Build_Model_New.ipynb` but on a
WIDER feature set: every `number_of_DQ*` and `percent*` trade feature in the
broad `all_open*` account groups (no niche sub-populations). The number
features are the ones the experian placeholder change moves, so this pair of
notebooks tests whether that change carries signal the percent-only
experiment couldn't see.

This notebook = **OLD (regular model-engine, shipping behavior: processed_old)**.
Counterpart: `Build_Model_NumberPercentOpen_New.ipynb` —
both freeze the identical feature list to
`Features_Number_And_Percent_Open.json`, so the only difference between the
two models is the aggregator behavior that produced the data.

Output -> `payment_processing_research_data/models/model_npo_old/`.
Model-engine kernel.

In [1]:
# Freeze the feature list: every number_of_DQ* and percent* feature in the
# broad 'all_open*' groups (no niche sub-populations). Written to its own
# json so the NEW and OLD notebooks train on the IDENTICAL list.
import os, json, glob
import pyarrow.parquet as pq
from configs import DATA_DIR

FEATURES_PATH = os.path.join(DATA_DIR, 'Features_Number_And_Percent_Open.json')

schema_file = sorted(glob.glob(os.path.join(
    DATA_DIR, 'samples', 'transunion_test', 'processed_new', 'part-*.parquet')))[0]
cols = pq.read_schema(schema_file).names   # schema only, no data load

def grp(c):
    return c.split('__')[-1]

number_feats  = [c for c in cols if 'number_of_DQ' in c and 'in_last' in c
                 and grp(c).startswith('all_open')]
percent_feats = [c for c in cols if ('percent_of_DQ' in c or 'percent_accounts_with_DQ' in c)
                 and 'in_last' in c and grp(c).startswith('all_open')]
features = number_feats + percent_feats

with open(FEATURES_PATH, 'w') as f:
    json.dump(features, f, indent=2)
print(f'{len(number_feats)} number + {len(percent_feats)} percent = '
      f'{len(features)} features -> {FEATURES_PATH}')

128 number + 160 percent = 288 features -> /home/jag/payment-processor-research/payment_processing_research_data/Features_Number_And_Percent_Open.json


In [2]:
import os, glob, json, importlib
import model_configs
importlib.reload(model_configs)
from model_engine.model_builder.build_model import build_model

VARIANT = 'npo_old'   # npo = Number + Percent, Open groups
out_dir = os.path.join(model_configs.MODELS_DIR, f'model_{VARIANT}')
os.makedirs(out_dir, exist_ok=True)
print('variant:', VARIANT, '| output ->', out_dir)

# OLD = regular model-engine (shipping behavior): processed_old everywhere.
asset = model_configs.build_asset('old')
print(f'trade files: {len(asset["data"]["trade"]["data"])}')

# train on the number+percent open feature list instead of the default
features = json.load(open(FEATURES_PATH))
asset['data']['trade']['io_params']['keep_features'] = features
asset['config']['base_features'] = FEATURES_PATH
print(f'keep_features: {len(features)} (number + percent, all_open* groups)')

/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



variant: npo_old | output -> /home/jag/payment-processor-research/payment_processing_research_data/models/model_npo_old
trade files: 6
keep_features: 288 (number + percent, all_open* groups)


In [3]:
# sanity: every table has files before we train
for tbl in ['app', 'trade', 'target']:
    n = len(asset['data'][tbl]['data'])
    print(f'{tbl:7}: {n} files')
    assert n > 0, f'no {tbl} files'
print('data_split:', asset['config']['data_split'])
json.dump(asset, open(os.path.join(out_dir, 'asset.json'), 'w'), indent=2)
print('wrote', os.path.join(out_dir, 'asset.json'))

app    : 6 files
trade  : 6 files
target : 6 files
data_split: {'train': {'start_date': '2019-04-01', 'end_date': '2019-07-01'}, 'test': {'start_date': '2019-07-01', 'end_date': '2020-04-01'}}
wrote /home/jag/payment-processor-research/payment_processing_research_data/models/model_npo_old/asset.json


In [4]:
import logging, traceback

log_path = os.path.join(out_dir, f'build_model_{VARIANT}.log')
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
root = logging.getLogger(); root.addHandler(fh); root.setLevel(logging.INFO)

try:
    build_model(asset, out_dir)
    root.info('BUILD SUCCEEDED -> %s', out_dir)
    print('\nDONE -> model artifacts in', out_dir)
    print(sorted(os.listdir(out_dir)))
except Exception as e:
    root.error('BUILD FAILED: %s\n%s', e, traceback.format_exc())
    print('BUILD FAILED -- see log:', log_path)
    raise
finally:
    root.removeHandler(fh); fh.close()
    print('log saved ->', log_path)

parsing model-builder asset


Configuring model builder


INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_asset>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <input_data>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <monotonic_constraints_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <add_default_monotonic_constraints>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <add_default_monotonic_constraints>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Executing InputArtifact <fe_version>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <data_split>...
INFO:zaml.artifact_engine.logger:Executing InputArtifact <train_sample_weight>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <train_sample_weight>, thus it will be omitt

building model


INFO:zaml.artifact_engine.logger:Finished <versions>, total time spent: 0:00:03.771772
INFO:zaml.artifact_engine.logger:Executing MonotonicConstraintsListParser <parsed_monotonic_constraints_list>...
INFO:zaml.artifact_engine.logger:Finished <parsed_monotonic_constraints_list>, total time spent: 0:00:00.735744
INFO:zaml.artifact_engine.logger:Executing SplitterArtifact <splitter>...
INFO:zaml.artifact_engine.logger:Finished <splitter>, total time spent: 0:00:00.000471
INFO:zaml.artifact_engine.logger:Executing DataArtifact <data>...
INFO:zaml.artifact_engine.logger:Finished <data>, total time spent: 0:00:00.000224
INFO:zaml.artifact_engine.logger:Executing ExclusionListParser <parsed_exclusion_list>...
INFO:zaml.artifact_engine.logger:Not all required inputs are available for optional artifact <parsed_exclusion_list>, thus it will be omitted.
INFO:zaml.artifact_engine.logger:Finished <parsed_exclusion_list>, total time spent: 0:00:00.000422
INFO:zaml.artifact_engine.logger:Executing Bi

...

INFO:zaml.artifact_engine.logger:Finished <test_fe_data>, total time spent: 0:00:10.636530
INFO:zaml.artifact_engine.logger:Executing StaticAssetArtifact <static_asset>...
INFO:zaml.artifact_engine.logger:Finished <static_asset>, total time spent: 0:00:00.184661
INFO:zaml.artifact_engine.logger:Executing TrainHistoryArtifact <train_history>...
INFO:zaml.artifact_engine.logger:Finished <train_history>, total time spent: 0:00:00.000283
INFO:zaml.artifact_engine.logger:Executing BestModelParamsArtifact <best_model_params>...
INFO:zaml.artifact_engine.logger:Finished <best_model_params>, total time spent: 0:00:00.000229
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <train_scores>...


-------------------------
Name: FillNA
Transformer type: FillNA
Number of features: 288
Time spent: 5.010s


INFO:zaml.artifact_engine.logger:Finished <train_scores>, total time spent: 0:00:02.002187
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <train_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <train_submodel_scores>, total time spent: 0:00:00.000299
INFO:zaml.artifact_engine.logger:Executing ScoresArtifact <test_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_scores>, total time spent: 0:00:02.017701
INFO:zaml.artifact_engine.logger:Executing SubmodelScoresArtifact <test_submodel_scores>...
INFO:zaml.artifact_engine.logger:Finished <test_submodel_scores>, total time spent: 0:00:00.000309
INFO:zaml.artifact_engine.logger:Executing CalibrationObjectArtifact <calibration_object>...


INFO:zaml.artifact_engine.logger:Finished <calibration_object>, total time spent: 0:00:24.480976
INFO:zaml.artifact_engine.logger:Executing FeatureDefinition <feature_definition>...
INFO:zaml.artifact_engine.logger:Finished <feature_definition>, total time spent: 0:00:00.017228
INFO:zaml.artifact_engine.logger:Executing KeyFactorMapping <key_factors_mapping>...
INFO:zaml.artifact_engine.logger:Finished <key_factors_mapping>, total time spent: 0:00:00.000558
INFO:zaml.artifact_engine.logger:Executing ValueBasedKeyFactorMapping <value_based_key_factor_mapping>...
INFO:zaml.artifact_engine.logger:Finished <value_based_key_factor_mapping>, total time spent: 0:00:00.000551
INFO:zaml.artifact_engine.logger:Executing FeatureImportanceArtifact <feature_importance>...


INFO:zaml.artifact_engine.logger:Finished <feature_importance>, total time spent: 0:00:04.494510
INFO:zaml.artifact_engine.logger:Executing SubmodelFeatureImportanceArtifact <submodel_feature_importance>...
INFO:zaml.artifact_en


DONE -> model artifacts in /home/jag/payment-processor-research/payment_processing_research_data/models/model_npo_old
['artifact_manifest.json', 'asset.json', 'best_model_params.json', 'build_model_npo_old.log', 'calibration_object.obj', 'feature_definition.parquet', 'feature_importance.parquet', 'fit_time_info.json', 'keep_features.json', 'key_factors_mapping.json', 'model.obj', 'model_strategy.json', 'mrm_pipeline.obj', 'parsed_monotonic_constraints_list.json', 'pipeline.obj', 'score_recalibration_mapping.json', 'splitter.obj', 'static_asset.json', 'test_app.parquet', 'test_auc.json', 'test_data_summary.json', 'test_fe_data.parquet', 'test_ks.json', 'test_scores.parquet', 'test_target.parquet', 'test_zest_scores.parquet', 'top_features.parquet', 'train_app.parquet', 'train_auc.json', 'train_data_summary.json', 'train_fe_data.parquet', 'train_history.json', 'train_ks.json', 'train_scores.parquet', 'train_target.parquet', 'train_zest_scores.parquet', 'value_based_key_factor_mapping.js

In [5]:
def load_json(path):
    return json.load(open(path))

In [6]:
path_old = '/home/jag/payment-processor-research/payment_processing_research_data/models/model_npo_old/test_auc.json'

In [7]:
path_new = '/home/jag/payment-processor-research/payment_processing_research_data/models/model_npo_new/test_auc.json'

In [8]:
load_json(path_old), load_json(path_new)

({'auc': 0.7218492195071672}, {'auc': 0.7217782024854063})